# Train spatial/non-spatial classifier

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
SEED = 42
np.random.seed(SEED)

## ✂ Split our manually labelled dataset into train/val/test (80/10/10)

In [3]:
data = (
    pd.read_excel('interim/is_spatial_20260125_edits.xlsx')
    .dropna(subset=['human_check'])
    .assign(
        y=lambda df_: df_.human_check.astype(int)
    )
    .filter(['query_id', 'query_text', 'y'])
)

print(data.y.value_counts())

data

y
0    755
1    718
Name: count, dtype: int64


,query_id,query_text,y
0,1135449,drugs that may increase homicidal thoughts,0
1,773756,what is motionless,0
2,349892,how to choose a diet plan,0
3,477401,population of carlsbad,0
4,1010534,which herb can heal bladder,0
...,...,...,...
11844,611868,what county is reynolds nd,1
11854,813149,what is the county for kermit tx,1
11898,1167689,weather in north dakota in may,1
11963,1163879,"what county is harvard, ma in/",1


In [4]:
# Split original data into train + test
train_val_df, test_df = train_test_split(
    data,
    test_size=0.1,
    random_state=SEED,
    stratify=data['y'] # ensure equal number of spatial/non-spatial
)

# Now let's split original train into train + val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.11111, # ca 10% of the original set
    random_state=SEED,
    stratify=train_val_df['y']
)

In [5]:
print('==Train counts==')
print(train_df.y.value_counts())

print('\n\n==Val counts==')
print(val_df.y.value_counts())

print('\n\n==Test counts==')
print(test_df.y.value_counts())

==Train counts==
y
0    603
1    574
Name: count, dtype: int64


==Val counts==
y
0    76
1    72
Name: count, dtype: int64


==Test counts==
y
0    76
1    72
Name: count, dtype: int64


In [6]:
test_df.y.value_counts()

y
0    76
1    72
Name: count, dtype: int64

In [7]:
train_df.to_csv('output/classifier.train.csv', index=False)
val_df.to_csv('output/classifier.val.csv', index=False)
test_df.to_csv('output/classifier.test.csv', index=False)

# 💪 Train our classifier

In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from setfit import Trainer, TrainingArguments, SetFitModel
from transformers import EarlyStoppingCallback
import torch

In [9]:
train_ds = Dataset.from_pandas(
    pd.read_csv('output/classifier.train.csv', usecols=['query_text', 'y'])
)

val_ds = Dataset.from_pandas(
    pd.read_csv('output/classifier.val.csv', usecols=['query_text', 'y'])
)

test_ds  = Dataset.from_pandas(
    pd.read_csv('output/classifier.test.csv', usecols=['query_text', 'y'])
)

In [11]:
model = SetFitModel.from_pretrained(
    'BAAI/bge-small-en-v1.5',
    num_pairs=100,
    num_iterations=10
)

# warning of initialising classification head with random weights is OK and expected!

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [13]:
args = TrainingArguments(
    num_epochs=1,
    batch_size=64,
    body_learning_rate=1e-5,
    seed=SEED,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=10,
    load_best_model_at_end=True,
    metric_for_best_model='embedding_loss',
    greater_is_better=False
)

def compute_metrics(preds, labels=None):
    # Called during training: preds = (y_pred, y_true)
    if labels is None:
        y_pred, y_true = preds
    else:
        y_pred, y_true = preds, labels

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    column_mapping={'query_text': 'text', 'y': 'label'},
    metric=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# disable pin_memory in the internal HF trainer
trainer.st_trainer.args.dataloader_pin_memory = False

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/1177 [00:00<?, ? examples/s]

In [14]:
trainer.train()

***** Running training *****
  Num unique pairs = 694262
  Batch size = 64
  Num epochs = 1


Step,Training Loss,Validation Loss
500,0.020000,0.057107
1000,0.001400,0.061707
1500,0.000800,0.068754
2000,0.000500,0.066344


/Users/ilya/miniconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:452: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(


In [15]:
results = trainer.evaluate(test_ds)
print(f"Accuracy: {results['accuracy']:.3f}")
print(f"F1: {results['f1']:.3f}")

Applying column mapping to the evaluation dataset
***** Running evaluation *****


Accuracy: 0.986
F1: 0.986


In [16]:
trainer.model.save_pretrained("spatial_classifier")

### Sence check: let's manually test on a few queries

In [17]:
model = SetFitModel.from_pretrained("spatial_classifier")

In [18]:
def is_spatial(q):
    pred = model.predict([q])
    print(f'{q}: {"✅" if pred[0] else "🚫"} \n---')

In [19]:
# clearly spatial
is_spatial('cafes within walking distance')
is_spatial('nearest hospital')
is_spatial('distance from paris to berlin')
is_spatial('where am i right now')
is_spatial('countries bordering ukraine')
is_spatial('elevation of mount fuji')
is_spatial('how far is the nearest bus stop')
is_spatial('route from oxford to cambridge')
is_spatial('restaurants along the m1')
is_spatial('flood risk in this area')

# clearly non-spatial (metaphorical / abstract)
is_spatial('close to my heart')
is_spatial('far from the truth')
is_spatial('a long way to go in my career')
is_spatial('high-level overview')
is_spatial('deep learning basics')
is_spatial('near impossible')
is_spatial('where do i stand politically')
is_spatial('on the edge emotionally')

# ambiguous / borderline (good test cases)
is_spatial('where should i live')              # needs disambiguation
is_spatial('how far can i go with this idea')  # metaphorical by default
is_spatial('where should i invest my money')   # abstract 'where'
is_spatial('how close are we to a solution')
is_spatial('what is my position on this')
is_spatial('where does this leave us')
is_spatial('how far apart are the classes')

# mixed spatial + non-spatial intent
is_spatial('how far is too far in relationships')
is_spatial('close friends who live far away')
is_spatial('where can i escape mentally')
is_spatial('distance learning programmes near me')

# tricky linguistic traps
is_spatial('where do i belong')
is_spatial('long way home')
is_spatial('keep your distance')
is_spatial('at a crossroads in life')
is_spatial('moving forward with the plan')

cafes within walking distance: ✅ 
---
nearest hospital: ✅ 
---
distance from paris to berlin: ✅ 
---
where am i right now: 🚫 
---
countries bordering ukraine: ✅ 
---
elevation of mount fuji: ✅ 
---
how far is the nearest bus stop: ✅ 
---
route from oxford to cambridge: ✅ 
---
restaurants along the m1: ✅ 
---
flood risk in this area: ✅ 
---
close to my heart: 🚫 
---
far from the truth: 🚫 
---
a long way to go in my career: 🚫 
---
high-level overview: 🚫 
---
deep learning basics: 🚫 
---
near impossible: 🚫 
---
where do i stand politically: 🚫 
---
on the edge emotionally: 🚫 
---
where should i live: ✅ 
---
how far can i go with this idea: 🚫 
---
where should i invest my money: 🚫 
---
how close are we to a solution: 🚫 
---
what is my position on this: 🚫 
---
where does this leave us: 🚫 
---
how far apart are the classes: 🚫 
---
how far is too far in relationships: 🚫 
---
close friends who live far away: 🚫 
---
where can i escape mentally: 🚫 
---
distance learning programmes near me: ✅ 
---

## Let's now train on the whole labelled set (incl. 80% train + 10% val + 10% test)

In [20]:
from datasets import concatenate_datasets
from setfit import SetFitModel, Trainer, TrainingArguments

In [21]:
full_ds = concatenate_datasets([train_ds, val_ds, test_ds])

In [22]:
model_prod = SetFitModel.from_pretrained(
    'BAAI/bge-small-en-v1.5',
    num_pairs=100,
    num_iterations=10
)

args_prod = TrainingArguments(
    num_epochs=1,
    batch_size=64,
    body_learning_rate=1e-5,
    seed=SEED,
    eval_strategy='no',
    save_strategy='no',
    logging_steps=500,
    load_best_model_at_end=False,
)

trainer_prod = Trainer(
    model=model_prod,
    args=args_prod, # using same args as above
    train_dataset=full_ds,
    eval_dataset=None,  # no validation - we are not tuning this time!
    column_mapping={'query_text': 'text', 'y': 'label'},
    metric=compute_metrics,
)

trainer_prod.st_trainer.args.dataloader_pin_memory = False
trainer_prod.train()

trainer_prod.model.save_pretrained('spatial_classifier_prod')

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset


Map:   0%|          | 0/1473 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 1087022
  Batch size = 64
  Num epochs = 1


Step,Training Loss
1,0.250700
500,0.180300
1000,0.013500
1500,0.002100
2000,0.001000
2500,0.000700
3000,0.000500
3500,0.000400
4000,0.000400
4500,0.000300


/Users/ilya/miniconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:452: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(


# Classify *all* MS MARCO queries

- Runs about 10 min (on an M3 processor)

In [23]:
from setfit import SetFitModel
import pandas as pd
from tqdm import tqdm

In [24]:
model = SetFitModel.from_pretrained("spatial_classifier")
queries = pd.read_csv("interim/queries.csv.zip")

# Define batch prediction function
def batch_predict(texts, batch_size=1024):
    preds = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        preds.extend(model.predict(batch))
    return preds #.item()

# Run prediction
queries['is_spatial_pred'] = batch_predict( queries['query_text'].tolist() )

100%|█████████████████████████████████████████| 988/988 [10:21<00:00,  1.59it/s]


In [25]:
queries

,query_id,query_text,is_spatial_pred
0,1048578,cost of endless pools/swim spa,tensor(0)
1,1048579,what is pcnt,tensor(0)
2,1048580,what is pcb waste,tensor(0)
3,1048581,what is pbis?,tensor(0)
4,1048582,what is paysky,tensor(0)
...,...,...,...
1010911,633855,what does canada post regulations mean,tensor(0)
1010912,1059728,wholesale lularoe price,tensor(0)
1010913,210839,how can i watch the day after,tensor(0)
1010914,908165,what to use instead of pgp in windows,tensor(0)


In [28]:
queries.is_spatial_pred = queries.is_spatial_pred.apply(lambda tsr: tsr.item())

In [29]:
queries.to_csv('output/queries-classified.csv.zip', index=False)